In [ ]:
# !conda install -c conda-forge eerepr

In [ ]:
import ee
# import geemap.core as geemap  # in colab
import geemap
import eerepr 

In [ ]:
# # Trigger the authentication flow.
# ee.Authenticate()

# # Initialize the library.
# ee.Initialize()

## Analysis constants and parameters

In [ ]:
# 10	
# #b6ff05	Grasslands: dominated by herbaceous annuals (<2m).

# 12	
# #c24f44	Croplands: at least 60% of area is cultivated cropland.

# 14	
# #ff6d4c	Cropland/Natural Vegetation Mosaics: mosaics of small-scale cultivation 40-60% with natural tree, shrub, or herbaceous vegetation.

In [ ]:
# Define a list of years
years = ee.List.sequence(2001, 2022)  # MODIS land cover data starts from 2001 !

# Define parameters
params = {
    'debug': False,
    'startDate': '2000-12-31',  # still  good to pre-filter 2001-2023 here,
    # because we do filterBounds() first before filtering by each year later in years.map()
    'endDate': '2024-01-01',
    'years': years,
    'exportTileScale': 1,  # 4 -> 1
    # 'selectLandCoverCode': [10],   # [10] for grassland ; [12, 14] for cropland  # !!!!!!!!!!!!!!!!!!!!!
     'selectLandCoverCode': [12, 14],  # do something quick for Guyo...
    'yearOfLcMaskForOneTimeData': 2022  # what year of land cover to use for one-time data??? Use latest year of the used MODIS land cover map i.e. 2022 as of 2024-03-23
}


In [ ]:
params["years"]

# Data

## Country boundaries

In [ ]:
countries = ee.FeatureCollection("users/hadicu06/Postdoc_Bonn/adminBoundaries/TM_WORLD_BORDERS-03")

In [ ]:
countries.size().getInfo()

In [ ]:
countries.first().propertyNames().getInfo()

In [ ]:
countries.propertyNames().getInfo()

### Remove Antarctica

In [ ]:
countries = countries.filter(ee.Filter.neq('ISO3', 'ATA'))

In [ ]:
countries.size().getInfo()

# Update params (countries polygons)

In [ ]:
params['countriesFc'] = countries

## MODIS land cover map

In [ ]:
modisLCData = (ee.ImageCollection('MODIS/061/MCD12Q1')
               .filterDate(params['startDate'], params['endDate']))

modisLC = modisLCData.select('LC_Type1')

## MODIS EVI

In [ ]:
modisEVIData = (ee.ImageCollection('MODIS/061/MOD13A3')
                .filterDate(params['startDate'], params['endDate'])
                .select(['EVI', 'SummaryQA']))

## MODIS NDVI

In [ ]:
modisNDVIData = (ee.ImageCollection('MODIS/061/MOD13A3')
                .filterDate(params['startDate'], params['endDate'])
                .select(['NDVI', 'SummaryQA']))

## Precipitation

In [ ]:
terraClimate = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
                .filterDate(params['startDate'], params['endDate']))

precipitation = terraClimate.select('pr')

## Temperature (land surface temperature)

In [ ]:
modisLSTData = (ee.ImageCollection('MODIS/061/MOD11A2')
                .filterDate(params['startDate'], params['endDate'])
                .select(['LST_Day_1km', 'QC_Day']))

## Minimum air temperature

In [ ]:
temperatureMin = terraClimate.select('tmmn')

## Maximum air temperature

In [ ]:
temperatureMax = terraClimate.select('tmmx')

## Solar radiation

In [ ]:
solarRadiation = terraClimate.select('srad')

## GMTED

In [ ]:
gmted = ee.Image('USGS/GMTED2010')
gmted_elev = gmted.select('be75').rename('gmted_elevation')

### Derive slope

In [ ]:
gmted_terrain = ee.Algorithms.Terrain(gmted_elev)
gmted_slope = gmted_terrain.select('slope').rename('gmted_slope')

In [ ]:
gmted_slope

## Geomorpho

### Terrain Ruggedness Index (TRI)

In [ ]:
geomorpho90m_tri = ee.ImageCollection("projects/sat-io/open-datasets/Geomorpho90m/tri")

### Roughness

In [ ]:
geomorpho90m_roughness = ee.ImageCollection("projects/sat-io/open-datasets/Geomorpho90m/roughness")

## MODIS NPP

In [ ]:
modisNPP = (ee.ImageCollection("MODIS/061/MOD17A3HGF").select('Npp')
            .filterDate(params['startDate'], params['endDate']))

## Add actual land degradation  (for Guyo)

"Hadi - for our policy paper, I need four data variables: soil erosion, soil compaction, soil carbon, soil water(moisture). Average country level"


In [ ]:
actualSoilOrganicCarbon = ee.Image("users/hadicu06/Postdoc_Bonn/actualLandDegradation/SOCS_0_30cm_year_2010AD_10km")
actualSoilDisplacement = ee.Image("users/hadicu06/Postdoc_Bonn/actualLandDegradation/SOIL_DISPLACEMENT_ESTIMATE_2019_yieldGapGrid")
actualSoilWater = ee.Image("users/hadicu06/Postdoc_Bonn/soilWater/combined_esacci_soil_moisture_2010_MEAN_sm")
soilCompactionDebt = ee.Image("users/hadicu06/Postdoc_Bonn/soilCompaction/Subsoil_compaction_susceptibility_index_SCSI").rename('Subsoil_compaction_susceptibility_index_mean')
actualSoilCompaction = soilCompactionDebt

## HaNi fertilizer (NH4), 2001 - 2019

In [ ]:
fertilizer = ee.ImageCollection("users/hadicu06/Postdoc_Bonn/fertilizer/HaNi_nfer_crop_nh4")
fertilizer = fertilizer.filter("time_year_int >= 2001 AND time_year_int <= 2019")

In [ ]:
def addSystemTimeProps(image):
    year = image.get('time_year_int')
    startDate = ee.Date.fromYMD(year, 1, 1)
    endDate = startDate.advance(1, 'year')
  
    startDateMillis = startDate.millis()
    endDateMillis = endDate.millis()
  
    return image.set({
        'system:time_start': startDateMillis,
        'system:time_end': endDateMillis,
    })


In [ ]:
fertilizer = fertilizer.map(addSystemTimeProps)

In [ ]:
fertilizer = fertilizer.map(lambda img: img.rename('fertilizer'))

In [ ]:
fertilizer

## Irrigation (2000, 2005, 2010)

In [ ]:
irrigation_2000 = ee.Image("users/hadicu06/Postdoc_Bonn/irrigation/HID_v10/HID_aei_ha__AEI_EARTHSTAT_CP_2000").rename('irrigation_2000')
irrigation_2005 = ee.Image("users/hadicu06/Postdoc_Bonn/irrigation/HID_v10/HID_aei_ha__AEI_EARTHSTAT_CP_2005").rename('irrigation_2005')
irrigation_2010 = ee.Image("users/hadicu06/Postdoc_Bonn/irrigation/G_AEI_2010").rename('irrigation_2010')

In [ ]:
irrigation = ee.ImageCollection.fromImages([
    # irrigation_2000.set('system:time_start', ee.Date.fromYMD(2000, 1, 1).millis()), 
    irrigation_2000.set('system:time_start', ee.Date.fromYMD(2001, 1, 1).millis()),     ## cause main analysis time series starts from 2001
    irrigation_2005.set('system:time_start', ee.Date.fromYMD(2005, 1, 1).millis()),
    irrigation_2010.set('system:time_start', ee.Date.fromYMD(2010, 1, 1).millis())
])

In [ ]:
irrigation = irrigation.map(lambda img: img.rename('irrigation'))

In [ ]:
irrigation

## Pesticide (2015)

In [ ]:
pesticide = ee.Image("users/hadicu06/Postdoc_Bonn/pesticide/APR_sum_2015_H").rename('pesticide')

## Params (reducers)

In [ ]:
params['reducerShortname'] = {
  'Max': ee.Reducer.max(),
  'Mean': ee.Reducer.mean(),
  'Sum': ee.Reducer.sum()
}

params['annualReducer'] = {
  'modisEVIData': 'Mean',   # Max -> Mean
  'modisNDVIData': 'Mean',  # Max -> Mean
  'precipitation': 'Sum',
  'temperatureMin': 'Mean',
  'temperatureMax': 'Mean',
  'modisLSTData': 'Mean',
  'solarRadiation': 'Mean',
  'modisNPP': 'Mean',
  'fertilizer': 'Mean',                 # data is one value per year so it's doing nothing here
  'irrigation': 'Mean',         # data is one value per year so it's doing nothing here
}

params['spatialReducer'] = {
  'modisEVIData': 'Mean',
  'modisNDVIData': 'Mean',
  'precipitation': 'Mean',
  'temperatureMin': 'Mean',
  'temperatureMax': 'Mean',
  'modisLSTData': 'Mean',
  'solarRadiation': 'Mean',
  'gmted_elev': 'Mean',
  'gmted_slope': 'Mean',
  'geomorpho90m_tri': 'Mean',
  'geomorpho90m_roughness': 'Mean',
  'modisNPP': 'Mean',
  # Add actual land degradation for Guyo
  'actualSoilOrganicCarbon': 'Mean',
  'actualSoilDisplacement': 'Mean',
  'actualSoilWater': 'Mean',
  'actualSoilCompaction': 'Mean',
  'fertilizer': 'Sum',
  'irrigation': 'Sum',
  'pesticide': 'Mean'
}

## Helper functions

In [ ]:
# Python code
# Helper function to extract the values from specific bits
# The input parameter can be an ee.Number() or ee.Image()
# Code adapted from https://gis.stackexchange.com/a/349401/5160
# From https://code.earthengine.google.co.in/e21636e7db4a3f757be2f51ed261719f
def bitwise_extract(input, from_bit, to_bit):
    mask_size = ee.Number(1).add(to_bit).subtract(from_bit)
    mask = ee.Number(1).leftShift(mask_size).subtract(1)
    return input.rightShift(from_bit).bitwiseAnd(mask)

### Functions for projection stuff

In [ ]:
def getNominalScale(ee_object):
    if isinstance(ee_object, ee.image.Image):
        scale = ee_object.projection().nominalScale().getInfo()        
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        scale = ee_object.first().projection().nominalScale().getInfo()
        
    return scale

In [ ]:
def getCrs(ee_object):
    if isinstance(ee_object, ee.image.Image):
        crs = ee_object.projection().getInfo()['crs']       
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        crs = ee_object.first().projection().getInfo()['crs']
    
    return crs

In [ ]:
def getTransform(ee_object):
    if isinstance(ee_object, ee.image.Image):
        transform = ee_object.projection().getInfo()['transform']      
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        transform = ee_object.first().projection().getInfo()['transform']
    
    return transform

In [ ]:
def getProj(ee_object):
    if isinstance(ee_object, ee.image.Image):
        proj = ee_object.projection()      
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        proj = ee_object.first().projection()  

    return proj 

## QA functions

In [ ]:
# Define a function to apply QA mask to MODIS EVI
def apply_qa_mask_modis_evi(image):
    evi = image.select('EVI')
    qa = image.select('SummaryQA')
    # SummaryQA = 0 "Good quality"
    qa_mask = qa.eq(0)
    mask = qa_mask
    return evi.updateMask(mask)

In [ ]:
# Define a function to apply QA mask to MODIS NDVI
def apply_qa_mask_modis_ndvi(image):
    ndvi = image.select('NDVI')
    qa = image.select('SummaryQA')
    # SummaryQA = 0 "Good quality"
    qa_mask = qa.eq(0)
    mask = qa_mask
    return ndvi.updateMask(mask)

In [ ]:
# Define a function to apply QA mask to MODIS LST
# Bits 0-1: Mandatory QA flags
# 0: Pixel produced, good quality, not necessary to examine more detailed QA
def apply_qa_mask_modis_lst(image):
    lst_day = image.select('LST_Day_1km')
    qc_day = image.select('QC_Day')
    qa_mask = bitwise_extract(qc_day, 0, 1).eq(0)
    mask = qa_mask
    return lst_day.updateMask(mask)

## Params (QA/QC functions)

In [ ]:

def do_nothing(image):
    return image

params['qaFilterFunc'] = {
  'modisEVIData': apply_qa_mask_modis_evi,
  'modisNDVIData': apply_qa_mask_modis_ndvi,
  'modisLSTData': apply_qa_mask_modis_lst,  
  'precipitation': do_nothing,
  'solarRadiation': do_nothing,
  'temperatureMin': do_nothing,
  'temperatureMax': do_nothing,
  # 'gmted': applyQaMaskGmtedElev,
  'gmted_elev': do_nothing,
  'gmted_slope': do_nothing,
  'srtmTopoDiv': do_nothing,
  'geomorpho90m_tri': do_nothing,
  'geomorpho90m_roughness': do_nothing,  
  'modisNPP': do_nothing,
  # do something for Guyo quick
  'actualSoilOrganicCarbon': do_nothing,
  'actualSoilDisplacement': do_nothing,
  'actualSoilWater': do_nothing,
  'actualSoilCompaction': do_nothing,
  'fertilizer': do_nothing,
  'irrigation': do_nothing,
  'pesticide': do_nothing
}


## Params (projection information)

In [ ]:

params['nativeScale'] = {
  'modisLCData': getNominalScale(modisLCData),
  'modisEVIData': getNominalScale(modisEVIData),
  'modisNDVIData': getNominalScale(modisNDVIData),
  'precipitation': getNominalScale(precipitation),
  'temperatureMin': getNominalScale(temperatureMin),
  'temperatureMax': getNominalScale(temperatureMax),
  'modisLSTData': getNominalScale(modisLSTData),
  'solarRadiation': getNominalScale(solarRadiation),
  'gmted_elev': getNominalScale(gmted_elev),
  'gmted_slope': getNominalScale(gmted_slope),
  'geomorpho90m_tri': getNominalScale(geomorpho90m_tri),
  'geomorpho90m_roughness': getNominalScale(geomorpho90m_roughness),
  'modisNPP': getNominalScale(modisNPP),
    # do something for Guyo quick
  'actualSoilOrganicCarbon': getNominalScale(actualSoilOrganicCarbon),
  'actualSoilDisplacement': getNominalScale(actualSoilDisplacement),
  'actualSoilWater': getNominalScale(actualSoilWater),
  'actualSoilCompaction': getNominalScale(actualSoilCompaction),
  'fertilizer': getNominalScale(fertilizer),
  'irrigation': getNominalScale(irrigation),
  'pesticide': getNominalScale(pesticide)
}


params['nativeCrs'] = {
  'modisLCData': getCrs(modisLCData),
  'modisEVIData': getCrs(modisEVIData),
  'modisNDVIData': getCrs(modisNDVIData),
  'precipitation': 'EPSG:4326',  # cause terraClimate only has wkt, but transform looks like lat-lon; not sure if Earth Engine accepts crs argument given as WKT string instead of EPSG code
  'temperatureMin': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'temperatureMax': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'modisLSTData': getCrs(modisLSTData),
  'solarRadiation': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'gmted_elev': getCrs(gmted_elev),
  'gmted_slope': getCrs(gmted_slope),
  'geomorpho90m_tri': getCrs(geomorpho90m_tri),
  'geomorpho90m_roughness': getCrs(geomorpho90m_roughness),
  'modisNPP': getCrs(modisNPP),
     # do something for Guyo quick
  'actualSoilOrganicCarbon': getCrs(actualSoilOrganicCarbon),
  'actualSoilDisplacement': getCrs(actualSoilDisplacement),
  'actualSoilWater': getCrs(actualSoilWater),
  'actualSoilCompaction': getCrs(actualSoilCompaction),
  'fertilizer': getCrs(fertilizer),
  'irrigation': getCrs(irrigation),
  'pesticide': getCrs(pesticide)
}


params['nativeTransform'] = {
  'modisLCData': getTransform(modisLCData),
  'modisEVIData': getTransform(modisEVIData),
  'modisNDVIData': getTransform(modisNDVIData),
  'precipitation': getTransform(precipitation),
  'temperatureMin': getTransform(temperatureMin),
  'temperatureMax':getTransform(temperatureMax),
  'modisLSTData': getTransform(modisLSTData),
  'solarRadiation': getTransform(solarRadiation),
  'gmted_elev': getTransform(gmted_elev),
  'gmted_slope': getTransform(gmted_slope),
  'geomorpho90m_tri': getTransform(geomorpho90m_tri),
  'geomorpho90m_roughness': getTransform(geomorpho90m_roughness),
  'modisNPP': getTransform(modisNPP),
       # do something for Guyo quick
  'actualSoilOrganicCarbon': getTransform(actualSoilOrganicCarbon),
  'actualSoilDisplacement': getTransform(actualSoilDisplacement),
  'actualSoilWater': getTransform(actualSoilWater),
  'actualSoilCompaction': getTransform(actualSoilCompaction),
  'fertilizer': getTransform(fertilizer),
  'irrigation': getTransform(irrigation),
  'pesticide': getTransform(pesticide)
}

params['nativeProj'] = {
  'modisLCData': getProj(modisLCData),
  'modisEVIData': getProj(modisEVIData),
  'modisNDVIData': getProj(modisNDVIData),
  'precipitation': getProj(precipitation),
  'temperatureMin': getProj(temperatureMin),
  'temperatureMax': getProj(temperatureMax),
  'modisLSTData': getProj(modisLSTData),
  'solarRadiation': getProj(solarRadiation),
  'gmted_elev': getProj(gmted_elev),
  'gmted_slope': getProj(gmted_slope),
  'geomorpho90m_tri': getProj(geomorpho90m_tri),
  'geomorpho90m_roughness': getProj(geomorpho90m_roughness),
  'modisNPP': getProj(modisNPP),
         # do something for Guyo quick
  'actualSoilOrganicCarbon': getProj(actualSoilOrganicCarbon),
  'actualSoilDisplacement': getProj(actualSoilDisplacement),
  'actualSoilWater': getProj(actualSoilWater),
  'actualSoilCompaction': getProj(actualSoilCompaction),
  'fertilizer': getProj(fertilizer),
  'irrigation': getProj(irrigation),
  'pesticide': getProj(pesticide)
}


## Params (whether data is time series)  

In [ ]:
params['data_is_time_series'] = {
  'modisEVIData': True,
  'modisNDVIData': True,
  'precipitation': True,
  'temperatureMin': True,
  'temperatureMax':True,
  'modisLSTData': True,
  'solarRadiation': True,
  'gmted_elev': False,
  'gmted_slope': False,
  'geomorpho90m_tri': False,
  'geomorpho90m_roughness': False,
  'modisNPP': True,  
   # do something for Guyo quick
  'actualSoilOrganicCarbon': False,
  'actualSoilDisplacement': False,
  'actualSoilWater': False,
  'actualSoilCompaction': False,
  'fertilizer': True,
  'irrigation': True,
  'pesticide': False
  }

# Export

### Design 1: by country, by variable, all countries

In [ ]:
print(countries.size().getInfo())

In [ ]:
params

In [ ]:
[1] * len(params['selectLandCoverCode'])

In [ ]:
params["yearOfLcMaskForOneTimeData"]

In [ ]:
def tempFunc(forcedInputScale=None):
    inputScale = 90
    if forcedInputScale is not None:
        inputScale = forcedInputScale
    return inputScale

print(f"tempFunc(): {tempFunc()}")

print(f"tempFunc(1000): {tempFunc(1000)}")


In [ ]:
src=fertilizer
variableId="irrigation"
outputVariableName="irrigation"
countriesFc=params['countriesFc']
exportDestinationMode="drive"
outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_06_27_Guyo/'  # not used
driveFolder='POSTDOC_BONN_GEE_v2'
debug=False  

In [ ]:
params['nativeScale']

In [ ]:
bla = params['nativeScale']["fertilizer"]
bla

In [ ]:
inputScale = params['nativeScale'][variableId]


inputCrs = params['nativeCrs'][variableId]
inputProj = params['nativeProj'][variableId]
inputTransform = params['nativeTransform'][variableId]

spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]
qaFilterFunc = params['qaFilterFunc'][variableId]

data_is_time_series = params['data_is_time_series'][variableId]  
    
if data_is_time_series:
    annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
else:
    annualReducer = None


outputStatsName = outputVariableName

if data_is_time_series:
    outputAssetNameBase = outputVariableName + "_annual" + params['annualReducer'][variableId] + "_spatial" + params['spatialReducer'][variableId]
else:
    outputAssetNameBase = outputVariableName  + "_spatial" + params['spatialReducer'][variableId]
    


In [ ]:
countryIds = countriesFc.distinct('ISO3').aggregate_array('ISO3').getInfo() 

In [ ]:
country = countries.filter(ee.Filter.eq("ISO3", "CCK")).first()

In [ ]:
geom = country.geometry()

geomSimple = geom.simplify(500)  # 1000 -> 500

# Filter bound the land cover (for MODIS land cover, one image per year)

modisLCBoundFilt = modisLC.filterBounds(geomSimple)  # here here


In [ ]:
year = 2006

In [ ]:
# ------- Filter bounds the image to reduce i.e. "src" --------      


srcBoundFilt = src.filterBounds(geomSimple)


# ------- Filter year --------       

startDate = ee.Date.fromYMD(year, 1, 1)
endDate = startDate.advance(1, 'year')
dateFilter = ee.Filter.date(startDate, endDate)


# ------- Mask out pixels that are not the land cover type of interest --------

modisLCYear = modisLCBoundFilt.filter(dateFilter).first()


landCoverMaskYear = modisLCYear.remap(params['selectLandCoverCode'], 
    [1] * len(params['selectLandCoverCode']) # here here # [value for _ in given_list] or [value] * len(given_list)
).rename('selectLandCover')      


srcYear = srcBoundFilt.filter(dateFilter)

def maskLandCover(image):
    return image.updateMask(landCoverMaskYear)


srcYearLCMasked = srcYear.map(maskLandCover)


# ------- Apply QA/QC mask --------

srcYearLCMaskedQaFilt = srcYearLCMasked.map(qaFilterFunc)


# ------- Apply temporal (annual) reducer as necessary --------

srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFilt.reduce(annualReducer)  

srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFiltAnnualReduced.rename(outputVariableName)


# ------- Apply spatial reducer --------

stats = srcYearLCMaskedQaFiltAnnualReduced.reduceRegion(
    reducer=spatialReducer,
    geometry=geomSimple,
    scale=inputScale,
    crs=inputCrs,
    # crsTransform= ,
    maxPixels=1e13,
    # tileScale=params['exportTileScale']
    tileScale=1
)


In [ ]:
print(stats.getInfo())

In [ ]:
# outputStatsName

In [ ]:
# stats.get(outputStatsName)

In [ ]:
# stats.keys().get(0)

In [ ]:
outputFeature = (ee.Feature(None)
            .copyProperties(country)          # because country can be a multipolygon
            .set('year', year, outputVariableName, stats.get(outputStatsName))) # # to check if using another string object as dictionary keyword works...


In [ ]:
outputFeature

### exportAllYearsByCountryByGivenVariable()

In [ ]:
def exportAllYearsByCountryByGivenVariable(
    src,
    variableId,
    outputVariableName,
    countriesFc,
    exportDestinationMode,
    outputAssetParent,
    driveFolder,
    exportTileScale=params['exportTileScale'],    # default
    geomSimplifyMeters=500,                        # default
    forcedInputScale=None,
    debug=False,
    years = params["years"]
):

    
        
    inputScale = params['nativeScale'][variableId]

    if forcedInputScale is not None:
        inputScale = forcedInputScale

    inputCrs = params['nativeCrs'][variableId]
    inputProj = params['nativeProj'][variableId]
    inputTransform = params['nativeTransform'][variableId]

    spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]
    qaFilterFunc = params['qaFilterFunc'][variableId]

    data_is_time_series = params['data_is_time_series'][variableId]  
     
    if data_is_time_series:
        annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
    else:
        annualReducer = None

    
    outputStatsName = outputVariableName
    
    if data_is_time_series:
        outputAssetNameBase = outputVariableName + "_annual" + params['annualReducer'][variableId] + "_spatial" + params['spatialReducer'][variableId]
    else:
        outputAssetNameBase = outputVariableName  + "_spatial" + params['spatialReducer'][variableId]
    

    counter = 0  
     
    countryIds = countriesFc.distinct('ISO3').aggregate_array('ISO3').getInfo()     # .getInfo() ! 

    for countryId in countryIds:   # for each region
    # for countryId in countryIdsRemain:   # if some countries failed, replace here manually for now
        
        counter += 1
        
        print("counter ", counter)
        
        print("processing ", countryId)

        # ------- Filter bounds the land cover for masks --------

        country = countries.filter(ee.Filter.eq("ISO3", countryId)).first()

        geom = country.geometry()

        geomSimple = geom.simplify(geomSimplifyMeters)  # 1000 -> 500

        # Filter bound the land cover (for MODIS land cover, one image per year)

        modisLCBoundFilt = modisLC.filterBounds(geomSimple)  # here here



         # ------- FUNCTION TO BE MAPPED OVER YEARS --------

        def processOneYear(year):

           # ------- Filter bounds the image to reduce i.e. "src" --------      


            srcBoundFilt = src.filterBounds(geomSimple)
           

            # ------- Filter year --------       

            startDate = ee.Date.fromYMD(year, 1, 1)
            endDate = startDate.advance(1, 'year')
            dateFilter = ee.Filter.date(startDate, endDate)


            # ------- Mask out pixels that are not the land cover type of interest --------

            modisLCYear = modisLCBoundFilt.filter(dateFilter).first()


            landCoverMaskYear = modisLCYear.remap(params['selectLandCoverCode'], 
               [1] * len(params['selectLandCoverCode']) # here here # [value for _ in given_list] or [value] * len(given_list)
            ).rename('selectLandCover')      


            srcYear = srcBoundFilt.filter(dateFilter)

            def maskLandCover(image):
                return image.updateMask(landCoverMaskYear)


            srcYearLCMasked = srcYear.map(maskLandCover)


            # ------- Apply QA/QC mask --------

            srcYearLCMaskedQaFilt = srcYearLCMasked.map(qaFilterFunc)


            # ------- Apply temporal (annual) reducer as necessary --------
     
            srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFilt.reduce(annualReducer)  

            srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFiltAnnualReduced.rename(outputVariableName)


            # ------- Apply spatial reducer --------

            stats = srcYearLCMaskedQaFiltAnnualReduced.reduceRegion(
                reducer=spatialReducer,
                geometry=geomSimple,
                scale=inputScale,
                crs=inputCrs,
                # crsTransform= ,
                maxPixels=1e13,
                # tileScale=params['exportTileScale']
                tileScale=exportTileScale
            )

            if debug:
                print("stats: ")
                print(stats.getInfo())
                # print("stats.get(outputStatsName): ")
                # print(stats.get(outputStatsName).getInfo())
            

            # make null geometry feature, just copy the properties from the country feature

            outputFeature = (ee.Feature(None)
                .copyProperties(country)          # because country can be a multipolygon
                # .set('year', year, outputVariableName, stats.get(outputStatsName))) # # to check if using another string object as dictionary keyword works...
                # .set('year', year, outputVariableName, stats.keys().get(0)))
                .set('year', year)
                .set(stats)
                )



            if debug:
                print("outputFeature: ")
                print(outputFeature.getInfo())
             
            return outputFeature    

       # ------- end of FUNCTION TO BE MAPPED OVER YEARS --------


         # ------- FUNCTION TO PROCESS ONE-TIME DATA --------

        def processOneTimeData(ee_object):  # accepts ee.Image or ee.ImageCollection 

            # ------- MODIS land cover to mask out land cover types not of interest --------
            # This is the only place where .filterDate() is relevant as "src" is one-time data

            startDate = ee.Date.fromYMD(params["yearOfLcMaskForOneTimeData"], 1, 1)   
            endDate = startDate.advance(1, 'year')
            modisLCYear = modisLCBoundFilt.filter(ee.Filter.date(startDate, endDate)).first()

            landCoverMaskYear = modisLCYear.remap(params['selectLandCoverCode'], 
               [1] * len(params['selectLandCoverCode']) # here here # [value for _ in given_list] or [value] * len(given_list)
            ).rename('selectLandCover')   

            def maskLandCover(image):
                return image.updateMask(landCoverMaskYear)

            # --------- If "src" is an image -----------------------
            
            if isinstance(src, ee.image.Image):
    
                # ------- Filter data by space -------- 

                srcSpaceFilt = src.clip(geomSimple)

                 # ------- Mask out land cover types not of interest  --------

                srcSpaceFiltLcMasked = maskLandCover(srcSpaceFilt)

                # ------- Apply QA/QC filter as necessary -------- 

                srcSpaceFiltLcMaskedQaFilt = qaFilterFunc(srcSpaceFiltLcMasked)            

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                # ------- Clip --------

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.clip(geomSimple)   ## maybe not necessary...


            # --------- If "src" is an image collection -----------------------

            elif isinstance(src, ee.imagecollection.ImageCollection):
                                
                # ------- Filter data by space -------- 

                srcSpaceFilt = src.filterBounds(geomSimple)

                 # ------- Mask out land cover types not of interest  --------

                srcSpaceFiltLcMasked = srcSpaceFilt.map(maskLandCover)

                # ------- Apply QA/QC filter as necessary -------- 

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMasked.map(qaFilterFunc)  

                # ------- Mosaic -------- 

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.mosaic()
                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                # ------- SetDefaultProjection and clip -------- 

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.setDefaultProjection(inputProj, inputTransform)

                srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.clip(geomSimple)   ## maybe not necessary...

        
            # ----- COMMON OPERATION --------

          
            # ------- Apply spatial reducer --------

            stats = srcSpaceFiltLcMaskedQaFilt.reduceRegion(
                reducer=spatialReducer,
                geometry=geomSimple,
                scale=inputScale,
                crs=inputCrs,
                # crsTransform= ,
                maxPixels=1e13,
                # tileScale=params['exportTileScale']
                tileScale=exportTileScale
            )            

            # make null geometry feature, just copy the properties from the country feature

            outputFeature = (ee.Feature(None)
                .copyProperties(country)          # because country can be a multipolygon
                .set(outputVariableName, stats.get(outputStatsName))) # # to check if using another string object as dictionary keyword works...


            return outputFeature    




       # ------- end of FUNCTION TO PROCESS ONE-TIME DATA --------

        
        if data_is_time_series:

            # ------ MAP THE FUNCTION OVER YEARS -------

            if debug:
                temp = processOneYear(2008)
                print("temp: ")
                print(temp.getInfo())

            if not debug:
                outputFc = (ee.FeatureCollection(
                    ee.List(years).map(processOneYear)
                    )
                ) 

        else:

            outputFc = ee.FeatureCollection([processOneTimeData(src)])   




      # ------- Batch export job --------

        outputAssetName = outputAssetNameBase + '_' + countryId

        propNamesToKeep = outputFc.first().propertyNames().remove(".geo").remove("system:index").getInfo()


        if exportDestinationMode == "asset":
            task = ee.batch.Export.table.toAsset(
                collection=outputFc,
                description=outputAssetName,
                assetId=outputAssetParent + outputAssetName
            )
        elif exportDestinationMode == "drive":
            task = ee.batch.Export.table.toDrive(
                collection=outputFc,
                description=outputAssetName,
                folder=driveFolder,
                fileFormat='CSV',
                selectors=propNamesToKeep
            )

        # ----- Start the export job! --------
        
        
        task.start()

        print(outputAssetName, "started")         


For demo:

In [ ]:
# def exportAllYearsByCountryByGivenVariable(
#     src,
#     variableId,
#     outputVariableName,
#     countriesFc,
#     exportDestinationMode,
#     outputAssetParent,
#     driveFolder,
#     exportTileScale=params['exportTileScale'],    
#     geomSimplifyMeters=500,                       
#     forcedInputScale=None
# ):
        
#     inputScale = params['nativeScale'][variableId]

#     if forcedInputScale is not None:
#         inputScale = forcedInputScale

#     inputCrs = params['nativeCrs'][variableId]
#     inputProj = params['nativeProj'][variableId]
#     inputTransform = params['nativeTransform'][variableId]

#     spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]
#     qaFilterFunc = params['qaFilterFunc'][variableId]

#     data_is_time_series = params['data_is_time_series'][variableId]  
     
#     if data_is_time_series:
#         annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
#     else:
#         annualReducer = None

    
#     outputStatsName = outputVariableName
    
#     if data_is_time_series:
#         outputAssetNameBase = outputVariableName + "_annual" + params['annualReducer'][variableId] + "_spatial" + params['spatialReducer'][variableId]
#     else:
#         outputAssetNameBase = outputVariableName  + "_spatial" + params['spatialReducer'][variableId]
    

#     counter = 0  
     
#     countryIds = countriesFc.distinct('ISO3').aggregate_array('ISO3').getInfo()     # .getInfo() ! 

#     for countryId in countryIds:   # for each region
#     # for countryId in countryIdsRemain:   # if some countries failed, replace here manually for now
        
#         counter += 1
        
#         print("counter ", counter)
        
#         print("processing ", countryId)

#         # ------- Filter bounds the land cover for masks --------

#         country = countries.filter(ee.Filter.eq("ISO3", countryId)).first()

#         geom = country.geometry()

#         geomSimple = geom.simplify(geomSimplifyMeters)  # 1000 -> 500

#         # Filter bound the land cover (for MODIS land cover, one image per year)

#         modisLCBoundFilt = modisLC.filterBounds(geomSimple)  # here here



#          # ------- FUNCTION TO BE MAPPED OVER YEARS --------

#         def processOneYear(year):

#            # ------- Filter bounds the image to reduce i.e. "src" --------      


#             srcBoundFilt = src.filterBounds(geomSimple)
           

#             # ------- Filter year --------       

#             startDate = ee.Date.fromYMD(year, 1, 1)
#             endDate = startDate.advance(1, 'year')
#             dateFilter = ee.Filter.date(startDate, endDate)


#             # ------- Mask out pixels that are not the land cover type of interest --------

#             modisLCYear = modisLCBoundFilt.filter(dateFilter).first()


#             landCoverMaskYear = modisLCYear.remap(params['selectLandCoverCode'], 
#                [1] * len(params['selectLandCoverCode']) # here here # [value for _ in given_list] or [value] * len(given_list)
#             ).rename('selectLandCover')      


#             srcYear = srcBoundFilt.filter(dateFilter)

#             def maskLandCover(image):
#                 return image.updateMask(landCoverMaskYear)


#             srcYearLCMasked = srcYear.map(maskLandCover)


#             # ------- Apply QA/QC mask --------

#             srcYearLCMaskedQaFilt = srcYearLCMasked.map(qaFilterFunc)


#             # ------- Apply temporal (annual) reducer as necessary --------
     
#             srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFilt.reduce(annualReducer)  

#             srcYearLCMaskedQaFiltAnnualReduced = srcYearLCMaskedQaFiltAnnualReduced.rename(outputVariableName)


#             # ------- Apply spatial reducer --------

            stats = srcYearLCMaskedQaFiltAnnualReduced.reduceRegion(
                reducer=spatialReducer,
                geometry=geomSimple,
                scale=inputScale,
                crs=inputCrs,
                # crsTransform= ,
                maxPixels=1e13,
                tileScale=exportTileScale
            )
            

#             # make null geometry feature, just copy the properties from the country feature

            outputFeature = (ee.Feature(None)
                .copyProperties(country)          # because country can be a multipolygon
                .set('year', year, outputVariableName, stats.get(outputStatsName))) 


#             return outputFeature    

#        # ------- end of FUNCTION TO BE MAPPED OVER YEARS --------


#          # ------- FUNCTION TO PROCESS ONE-TIME DATA --------

#         def processOneTimeData(ee_object):  # accepts ee.Image or ee.ImageCollection 

#             # ------- MODIS land cover to mask out land cover types not of interest --------
#             # This is the only place where .filterDate() is relevant as "src" is one-time data

#             startDate = ee.Date.fromYMD(params["yearOfLcMaskForOneTimeData"], 1, 1)   
#             endDate = startDate.advance(1, 'year')
#             modisLCYear = modisLCBoundFilt.filter(ee.Filter.date(startDate, endDate)).first()

#             landCoverMaskYear = modisLCYear.remap(params['selectLandCoverCode'], 
#                [1] * len(params['selectLandCoverCode']) # here here # [value for _ in given_list] or [value] * len(given_list)
#             ).rename('selectLandCover')   

#             def maskLandCover(image):
#                 return image.updateMask(landCoverMaskYear)

#             # --------- If "src" is an image -----------------------
            
#             if isinstance(src, ee.image.Image):
    
#                 # ------- Filter data by space -------- 

#                 srcSpaceFilt = src.clip(geomSimple)

#                  # ------- Mask out land cover types not of interest  --------

#                 srcSpaceFiltLcMasked = maskLandCover(srcSpaceFilt)

#                 # ------- Apply QA/QC filter as necessary -------- 

#                 srcSpaceFiltLcMaskedQaFilt = qaFilterFunc(srcSpaceFiltLcMasked)            

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

#                 # ------- Clip --------

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.clip(geomSimple)   ## maybe not necessary...


#             # --------- If "src" is an image collection -----------------------

#             elif isinstance(src, ee.imagecollection.ImageCollection):
                                
#                 # ------- Filter data by space -------- 

#                 srcSpaceFilt = src.filterBounds(geomSimple)

#                  # ------- Mask out land cover types not of interest  --------

#                 srcSpaceFiltLcMasked = srcSpaceFilt.map(maskLandCover)

#                 # ------- Apply QA/QC filter as necessary -------- 

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMasked.map(qaFilterFunc)  

#                 # ------- Mosaic -------- 

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.mosaic()
#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

#                 # ------- SetDefaultProjection and clip -------- 

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.setDefaultProjection(inputProj, inputTransform)

#                 srcSpaceFiltLcMaskedQaFilt = srcSpaceFiltLcMaskedQaFilt.clip(geomSimple)   ## maybe not necessary...

        
#             # ----- COMMON OPERATION --------

          
#             # ------- Apply spatial reducer --------

#             stats = srcSpaceFiltLcMaskedQaFilt.reduceRegion(
#                 reducer=spatialReducer,
#                 geometry=geomSimple,
#                 scale=inputScale,
#                 crs=inputCrs,
#                 # crsTransform= ,
#                 maxPixels=1e13,
#                 # tileScale=params['exportTileScale']
#                 tileScale=exportTileScale
#             )            

#             # make null geometry feature, just copy the properties from the country feature

#             outputFeature = (ee.Feature(None)
#                 .copyProperties(country)          # because country can be a multipolygon
#                 .set(outputVariableName, stats.get(outputStatsName))) # # to check if using another string object as dictionary keyword works...


#             return outputFeature    




#        # ------- end of FUNCTION TO PROCESS ONE-TIME DATA --------

        
#         if data_is_time_series:

#             # ------ MAP THE FUNCTION OVER YEARS -------

            outputFc = (ee.FeatureCollection(
                ee.List(params["years"]).map(processOneYear)
                )
            ) 

#         else:

#             outputFc = ee.FeatureCollection([processOneTimeData(src)])   




#       # ------- Batch export job --------

#         outputAssetName = outputAssetNameBase + '_' + countryId

#         propNamesToKeep = outputFc.first().propertyNames().remove(".geo").remove("system:index").getInfo()


#         if exportDestinationMode == "asset":
#             task = ee.batch.Export.table.toAsset(
#                 collection=outputFc,
#                 description=outputAssetName,
#                 assetId=outputAssetParent + outputAssetName
#             )
#         elif exportDestinationMode == "drive":
#             task = ee.batch.Export.table.toDrive(
#                 collection=outputFc,
#                 description=outputAssetName,
#                 folder=driveFolder,
#                 fileFormat='CSV',
#                 selectors=propNamesToKeep
#             )

#         # ----- Start the export job! --------
        
        
#         task.start()

#         print(outputAssetName, "started")         


## Testing

In [ ]:
params["years"]

In [ ]:
#   ## If debug mode

# # params['debug'] = True

# if params['debug']:
#     # params["years"] = [2001, 2010]
#     params["years"] = [2002]
# else:
#     params["years"] = years


# if params['debug']:
#     # params['countriesFc'] = countries.filter(ee.Filter.inList("ISO3", ['LIE', 'MLT']))
#     params['countriesFc'] = countries.filter(ee.Filter.inList("ISO3", ['CCK']))
# else:
# 	params['countriesFc'] = countries

In [ ]:
#  params['countriesFc'].size().getInfo()

In [ ]:
# !earthengine create folder users/hadicu06/Postdoc_Bonn/analysis/countryAverage/2024_03_23_Kirara

# !earthengine create folder users/hadicu06/Postdoc_Bonn/analysis/countryAverage/2024_03_23_Kirara_test

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisEVIData,
#     variableId='modisEVIData',
#     outputVariableName="evi",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNDVIData,
#     variableId='modisNDVIData',
#     outputVariableName="ndvi",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisLSTData,
#     variableId='modisLSTData',
#     outputVariableName="lst",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )


In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=precipitation,
#     variableId='precipitation',
#     outputVariableName="prec",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=solarRadiation,
#     variableId='solarRadiation',
#     outputVariableName="srad",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=temperatureMin,
#     variableId='temperatureMin',
#     outputVariableName="tmin",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=temperatureMax,
#     variableId='temperatureMax',
#     outputVariableName="tmax",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=gmted_elev,
#     variableId='gmted_elev',
#     outputVariableName="elev",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=gmted_slope,
#     variableId='gmted_slope',
#     outputVariableName="slope",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNPP,
#     variableId='modisNPP',
#     outputVariableName="npp",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara_test/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

Checked in Drive, looks OK!

# Big run!

## 1. modisEVIData

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=modisEVIData,
    variableId='modisEVIData',
    outputVariableName="evi",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
    driveFolder='POSTDOC_BONN_GEE',
    #exportTileScale=16   
    #geomSimplifyMeters=1000   
)

# ## running -> error when reached "CAN"

Failed for 'CAN': EEException: Computation timed out


In [ ]:
countryIds = params["countriesFc"].distinct('ISO3').aggregate_array('ISO3').getInfo()  

countryIds.index('CAN')

In [ ]:
# countryIdsRemain = countryIds[115:]

# print(countryIdsRemain)

In [ ]:
# params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIdsRemain))

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisEVIData,
#     variableId='modisEVIData',
#     outputVariableName="evi",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIdsRemain)),   ## !!!
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# ) 
# ## running -> all OK done


Still need to run for "CAN"

Run for "CAN", try:
- `tileScale=4` (computation timed out)
- `tileScale=16` (still computation timed out)
- simplify geometry 1000m (task successfully submittd -> ...)

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=modisEVIData,
    variableId='modisEVIData',
    outputVariableName="evi",
    countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', ['CAN'])),   ## !!!
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
    driveFolder='POSTDOC_BONN_GEE',
    # exportTileScale=16   ## 
    geomSimplifyMeters=1000
) 

## running -> OK done
## downloaded

## 2. modisNDVIData

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNDVIData,
#     variableId='modisNDVIData',
#     outputVariableName="ndvi",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

## Kirara is running
## all done OK
## to download from her google drive

## 3. modisLSTData

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisLSTData,
#     variableId='modisLSTData',
#     outputVariableName="lst",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

# ## running -> all OK done
## downloaded

## 4. Precipitation

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=precipitation,
#     variableId='precipitation',
#     outputVariableName="prec",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )

## running, all OK done
## downloaded

## 5. solarRadiation

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=solarRadiation,
#     variableId='solarRadiation',
#     outputVariableName="srad",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## running -> all OK done
## downloaded

## 6. temperatureMin

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=temperatureMin,
#     variableId='temperatureMin',
#     outputVariableName="tmin",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## running -> all OK done
## downloaded

## 7. temperatureMax

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=temperatureMax,
    variableId='temperatureMax',
    outputVariableName="tmax",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
    driveFolder='POSTDOC_BONN_GEE'   
)  

## running -> all OK done
## downloaded

## 8. gmted_elev

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=gmted_elev,
#     variableId='gmted_elev',
#     outputVariableName="elev",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## Kirara will run
## Downloaded

## 9. gmted_slope

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=gmted_slope,
#     variableId='gmted_slope',
#     outputVariableName="slope",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## Kirara will run
## Downloaded

## 10. geomorpho90m_roughness

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

# ## running -> error at "NIU"
## Downloaded

- processing  NIU
    - EEException: Dictionary.get: Dictionary does not contain key: rough.


In [ ]:
countryIds.index('NIU')

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIds[85:])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## Error when reached "CAN": EEException: Computation timed out.
## Downloaded


In [ ]:
countryIds.index("CAN")

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIds[115:])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## Error when reached "USA" : EEException: Computation timed out.
## Downloaded

In [ ]:
countryIds.index("USA")

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIds[117:])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## Error at "RUS"
## Downloaded

In [ ]:
countryIds.index("RUS")

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=geomorpho90m_roughness,
#     variableId='geomorpho90m_roughness',
#     outputVariableName="rough",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIds[219:])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## all OK from here
## downloaded

Still need to do "NIU", "CAN", "USA", "RUS"

Run for "NIU", "CAN", "USA", "RUS", try:
- ` geomSimplifyMeters=1000 ` : EEException: Computation timed out
- ` geomSimplifyMeters=1000 ` and ` exportTileScale=4 ` : still timed out
- ` geomSimplifyMeters=10000 ` succceeded for CAN and USA, but failed for RUS (Computation timed out.)
- ` geomSimplifyMeters=20000 ` still failed for RUS (Computation timed out.)
- ` geomSimplifyMeters=1000 ` and ` forcedInputScale=1000 ` : succeeded for all CAN, USA, RUS.
- ` geomSimplifyMeters=1000 ` and ` forcedInputScale=270 ` : succeeded for all CAN, USA, RUS. <- use this result in the end


In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=geomorpho90m_roughness,
    variableId='geomorpho90m_roughness',
    outputVariableName="rough",
    countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', ["CAN", "USA", "RUS"])), # !!  # "NIU" error: EEException: Dictionary.get: Dictionary does not contain key: rough.
    # countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', ["RUS"])), #  "CAN" and "USA" succeeded with geomSimplifyMeters=10000 , "RUS" still failed   
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
    driveFolder='POSTDOC_BONN_GEE_v2',  # !!
    # exportTileScale=4,                  # !!
    # geomSimplifyMeters=10000             # !!
    # geomSimplifyMeters=20000             # !!
    # geomSimplifyMeters=30000  
    geomSimplifyMeters=1000, 
    # forcedInputScale=1000
    forcedInputScale=270
)  

## running -> 

## 11. modisNPP

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNPP,
#     variableId='modisNPP',
#     outputVariableName="npp",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  


## Error until "USA": EEException: Too many concurrent aggregations.
## Downloaded

In [ ]:
countryIds.index("USA")

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNPP,
#     variableId='modisNPP',
#     outputVariableName="npp",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIds[117:])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE'   
# )  

## All done here
## Downloaded

- Still need to do "USA"

Run for "USA", try:
- `geomSimplifyMeters=1000` : 

In [ ]:
# exportAllYearsByCountryByGivenVariable(
#     src=modisNPP,
#     variableId='modisNPP',
#     outputVariableName="npp",
#     countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', ['USA'])),
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE_v2',  # !!
#     # exportTileScale=4,
#     geomSimplifyMeters=1000   
# )  

## running -> OK done
## downloaded

## For Guyo

### actualSoilOrganicCarbon 

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilOrganicCarbon,
    variableId='actualSoilOrganicCarbon',
    outputVariableName="actualSoilOrganicCarbon",
    countriesFc=params['countriesFc']
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'   
)  

# to run

### actualSoilDisplacement 

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilDisplacement  ,
    variableId='actualSoilDisplacement',
    outputVariableName="actualSoilDisplaceme",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'   
)  

# to run

### actualSoilWater 

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilWater   ,
    variableId='actualSoilWater',
    outputVariableName="actualSoilWater",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'   
)  

# to run

Failed until CAN cause disconnected. Need to run from CAN.

In [ ]:
countryIds = params["countriesFc"].distinct('ISO3').aggregate_array('ISO3').getInfo()  

countryIds.index('CAN')

In [ ]:
countryIdsRemain = countryIds[114:]

print(countryIdsRemain)

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilWater,
    variableId='actualSoilWater',
    outputVariableName="actualSoilWater",
    countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', countryIdsRemain)),
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'   
)  


RUS is missing somehow

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilWater,
    variableId='actualSoilWater',
    outputVariableName="actualSoilWater",
    countriesFc=params['countriesFc'].filter(ee.Filter.inList('ISO3', ['RUS'])),
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    # driveFolder='POSTDOC_BONN_GEE_v2'   
    driveFolder='POSTDOC_BONN_GEE'  

)  


### actualSoilCompaction 

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=actualSoilCompaction,
    variableId='actualSoilCompaction',
    outputVariableName="actualSoilCompaction",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_04_26_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'  
)  

# to 

### HaNi fertilizer (NH4), 2001 - 2019

In [ ]:
# exportAllYearsByCountryByGivenVariable(    ## just testing if previous code works
#     src=precipitation,
#     variableId='precipitation',
#     outputVariableName="prec",
#     countriesFc=params['countriesFc'],
#     exportDestinationMode="drive",
#     outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_03_23_Kirara/',
#     driveFolder='POSTDOC_BONN_GEE_v2'   
# )
# ## yes it does!

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=fertilizer,
    variableId='fertilizer',
    outputVariableName="fertilizer",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_06_27_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2',
    debug=False  
)  


### Irrigation (2000, 2005, 2010)

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=irrigation,
    variableId="irrigation",
    outputVariableName="irrigation",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_06_27_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2',
    debug=False,
    years=ee.List([2001, 2005, 2010]) 
)  

### Pesticide (2015)

In [ ]:
exportAllYearsByCountryByGivenVariable(
    src=pesticide,
    variableId='pesticide',
    outputVariableName="pesticide",
    countriesFc=params['countriesFc'],
    exportDestinationMode="drive",
    outputAssetParent='users/hadicu06/Postdoc_Bonn/analysis/countryAverage/' + '2024_06_27_Guyo/',  # not used
    driveFolder='POSTDOC_BONN_GEE_v2'  
)  
